In [ ]:
#1 class validation without pydantic
class Product:
    name: str
    price: float
    stock: int

    def __init__(self,name,price,stock):
        
        if not isinstance(name, str):
            raise ValueError(f"Incorrect type, expected str, but got: {type(name).__name__}")
        if not isinstance(price, float):
            raise ValueError(f"Incorrect type, expected float, but got: {type(price).__name__} ")
        if not isinstance(stock, int):
            raise ValueError(f"Incorrect type, expected int, but got: {type(stock).__name__}")

        self.name = name
        self.price = price
        self.stock = stock
    
try:
    product = Product(name="Skate", price=10.5, stock=1)
except Exception as e:
    print(e)
else:
    print(product.name)

Skate


In [ ]:
#2 basic BaseModel
from pydantic import BaseModel
class Book(BaseModel):
    title: str
    pages: int
    author: str

book = Book(title="Python Book", pages="350", author="Leonardo K")
print(book)

{"title":"Python Book","pages":350,"author":"Leonardo K"}


In [ ]:
#3 helper methods
from pydantic import BaseModel
from typing import Optional
class Book(BaseModel):
    name: str
    pages: int
    author: Optional[str] = None
    
book = Book(name="Default Book", pages=350)
print(book)

# model_fields_set: tracks which fields were explicitly provided during initialization ---> name and pages
print(book.model_fields_set)


# model_dump: returns a dictionary representation of the model
print(book.model_dump())

# model_dump_json(): returns a json of the model
print(book.model_dump_json())

# model_json_schema(): returns a json with more information
print(book.model_json_schema())

name='Default Book' pages=350 author=None
{'pages', 'name'}
{'name': 'Default Book', 'pages': 350, 'author': None}
{"name":"Default Book","pages":350,"author":null}
{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'pages': {'title': 'Pages', 'type': 'integer'}, 'author': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Author'}}, 'required': ['name', 'pages'], 'title': 'Book', 'type': 'object'}


In [ ]:
#4 nested models

from pydantic import BaseModel
from typing import List, Optional

class Item(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    items: List[Item]
    discount: Optional[float] = None
    
try:
    
    item_1_json = {"name":"Keyboard","price":10.0}
    item_2_json = {"name":"Mouse","price":14.5}

    item_1 = Item.model_validate(item_1_json)
    item_2 = Item.model_validate(item_2_json)

    order = Order(items=[item_1,item_2], discount=7.5)

except Exception as e:
    print(e)
else:
    print(order.model_dump_json())
    

{"items":[{"name":"Keyboard","price":10.0},{"name":"Mouse","price":14.5}],"discount":7.5}


In [46]:
#5 pydantic types (conlist, Field, HttpUrl, EmailStr, PositiveInt)

from pydantic import BaseModel, HttpUrl, EmailStr, PositiveInt, conlist, Field
from typing import List, Optional

class Department(BaseModel):
    name: str

class Company(BaseModel):
    name: str = Field(pattern=r'^[a-zA-Z]+$')
    email: EmailStr
    website: HttpUrl
    employees: PositiveInt
    departments: conlist(Department, min_length=1)

try:
    department = Department.model_validate({"name":"Infra"})
    
    company_json = {"name":"AltsCode","email":"altscode@web.com","website":"https://altscode.com","employees":1, "departments": [department]}
    
    company = Company.model_validate(company_json)
    
except Exception as e:
    print(e)
else:
    print(company.model_dump_json())



{"name":"AltsCode","email":"altscode@web.com","website":"https://altscode.com/","employees":1,"departments":[{"name":"Infra"}]}


In [50]:
#6 field validator
from pydantic import BaseModel, field_validator, ValidationError
class Profile(BaseModel):
    username: str
    age: int
    
    @field_validator('username')
    @classmethod
    def validate_username(cls, v):
        if len(v) < 3:
            raise ValueError("The username must contain at least 3 characters")
        return v.lower()
    
try:
    profile_json = {"username":"leok123","age":19}
    profile = Profile.model_validate(profile_json)

except ValidationError as e:
    print(e)
else:
    print(profile)

username='leok123' age=19


In [55]:
#7 model_validator
from pydantic import BaseModel, model_validator, ValidationError

class PasswordChange(BaseModel):
    password: str
    confirm_password: str
    
    @model_validator(mode="after")
    def check_passwords(self):
        if self.password != self.confirm_password:
            raise ValueError("Both passwords should be equal")
        return self

try:
    password_change_json = {"password":"leok123","confirm_password":"leok123"}
    password_change = PasswordChange.model_validate(password_change_json)

except ValidationError as e:
    print(e)
else:
    print(password_change)



password='leok123' confirm_password='leok123'


In [62]:
#8 field with alias, ge, gt, default_factory
from pydantic import BaseModel, model_validator, ValidationError,Field
from uuid import uuid4

class Transaction(BaseModel):
    id: str = Field(default_factory=lambda:uuid4().hex)
    amount: float = Field(..., gt=0)
    currency: str = Field(..., alias="currency_code", max_length=3)

try:
    transaction_json = {"amount":10.5,"currency_code":"brl"}
    transaction = Transaction.model_validate(transaction_json)
    
except ValidationError as e:
    print(e)
else:
    print(transaction.model_dump_json(by_alias=True))

{"id":"c50a46d1d2ea442c8794ae796c756f1b","amount":10.5,"currency_code":"brl"}


In [ ]:
#9 computed_fields & dataclass
from pydantic import BaseModel, ValidationError, computed_field

class Circle(BaseModel):
    radius: float
    
    @computed_field
    @property
    def area(self) -> float:
        pi = 3.14
        area = pi * self.radius
        return round(area, 2)

try:
    circle_json = {"radius":3.0}
    circle = Circle.model_validate(circle_json)
    
except ValidationError as e:
    print(e)
else:
    print(circle.model_dump_json())


{"radius":3.0,"area":9.42}


In [76]:
#10 strict mode & BaseSettings
from pydantic import BaseModel, ValidationError,SecretStr, Field, AliasChoices
from pydantic_settings import SettingsConfigDict, BaseSettings

class AppConfig(BaseSettings):
    model_config = SettingsConfigDict(env_file="env_practice", extra="ignore")
    database_url: SecretStr = Field(alias="DATABASE_URL",validation_alias=AliasChoices("database_url", "db_url"))
    debug: bool = Field(alias="DEBUG")

try:
    app_config = AppConfig()
    
except ValidationError as e:
    print(e)
else:
    print(app_config.model_dump_json())
    print(f"database url: {app_config.database_url.get_secret_value()}")

{"database_url":"**********","debug":true}
database url: 127.0.0.1
